# Step 12 – Second event: October 2025 floods (pure transfer test)
The models trained and fine-tuned on the **2024** data are applied, unchanged, to the **3–5 Oct 2025** floods.

| Area | Sentinel-1 flood / pre-flood | Sentinel-2 labels | Test |
|---|---|---|---|
| Koshi Terai | 8 Oct 2025 / 26 Sep 2025, orbit 12 | **8 Oct 2025 (same day)** | flood mapping accuracy + exposure |
| Kathmandu | 9 Oct 2025 / 27 Sep 2025, orbit 19 | 11 Oct 2025 | water accuracy + false alarms in steep terrain |

Models: A and B (original, 3 seeds each) and A and B **fine-tuned on 2024** (3 seeds each), plus threshold and change-detection baselines.

Runtime: **GPU (T4)**, about 25–40 min (most of it the Earth Engine export). Needs Earth Engine and Drive.

In [ ]:
PROJECT = 'your-ee-project-id'   # <-- your Earth Engine Cloud project ID
!pip -q install segmentation-models-pytorch rasterio
import os, math, time, subprocess, numpy as np, pandas as pd, rasterio, torch, ee
import segmentation_models_pytorch as smp, matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling
from scipy.ndimage import binary_erosion, binary_dilation, label as cc_label
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEV)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Paper2_flood'
NEPAL = '/content/drive/MyDrive/Paper2_flood_nepal'
OUT = f'{BASE}/12_event_2025'; os.makedirs(OUT, exist_ok=True)

ee.Authenticate(); ee.Initialize(project=PROJECT)
EVENTS = {
    'koshi_terai': dict(bbox=[86.85, 26.45, 87.15, 26.75], flood='2025-10-08', pre='2025-09-26', orbit=12, s2='2025-10-08'),
    'kathmandu':   dict(bbox=[85.20, 27.58, 85.48, 27.78], flood='2025-10-09', pre='2025-09-27', orbit=19, s2='2025-10-11'),
}
WEIGHTS = {}
for m in ['A_sar', 'B_sar_terrain']:
    WEIGHTS[(m, 'original', 0)] = f'{BASE}/02_terrain/unet_{m}.pt'
    for s in [1, 2]: WEIGHTS[(m, 'original', s)] = f'{BASE}/05_multiseed/runs/unet_{m}_seed{s}.pt'
    for s in [0, 1, 2]: WEIGHTS[(m, 'fine-tuned 2024', s)] = f'{BASE}/08_finetune/unet_{m}_seed{s}_finetuned.pt'
for f in WEIGHTS.values(): assert os.path.exists(f), f'missing {f}'
CH = {'A_sar': [0, 1], 'B_sar_terrain': [0, 1, 2, 3]}
print('ready')

## 1. Build and export one stack per area
Bands: `flood_VV, flood_VH, pre_VV, pre_VH, slope, hand, ls_mask, s2_water, dry_water`
- `s2_water`: Sentinel-2 MNDWI label on the label date (1 water, 0 dry, 255 cloud/ambiguous)
- `dry_water`: same label from the Feb–Apr 2025 Sentinel-2 median (normal water)

In [ ]:
DEG = math.pi / 180
dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic().setDefaultProjection('EPSG:4326', None, 30)
slope = ee.Terrain.slope(dem); aspect = ee.Terrain.aspect(dem)
hand = ee.Image('MERIT/Hydro/v1_0_1').select('hnd').unmask(0)
S2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

def acq(aoi, date, orbit):
    d = ee.Date(date)
    col = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi).filterDate(d, d.advance(1, 'day'))
           .filter(ee.Filter.eq('instrumentMode', 'IW')).filter(ee.Filter.eq('relativeOrbitNumber_start', orbit))
           .select(['VV', 'VH', 'angle']))
    return col.mosaic().setDefaultProjection(col.first().select('VV').projection()).clip(aoi)

def layover_shadow(img, aoi):
    theta = img.select('angle')
    look = ee.Number(ee.Terrain.aspect(theta).reduceRegion(ee.Reducer.mean(), aoi, 1000).get('aspect'))
    alpha = slope.multiply(DEG).tan().multiply(aspect.subtract(look).multiply(DEG).cos()).atan().divide(DEG)
    return alpha.multiply(-1).gt(theta).add(alpha.gt(ee.Image(90).subtract(theta)).multiply(2)).rename('ls_mask')

def clear(im):
    scl = im.select('SCL'); return scl.eq(2).Or(scl.gte(4).And(scl.lte(7)))
def lab(index, ok):
    return ee.Image(255).where(index.gt(0.1), 1).where(index.lt(-0.1), 0).updateMask(ok).unmask(255)

FILES = {}
tasks = []
for name, e in EVENTS.items():
    aoi = ee.Geometry.Rectangle(e['bbox'])
    fname = f"{name}_2025_flood{e['flood']}_pre{e['pre']}_orbit{e['orbit']}_s2{e['s2']}"
    FILES[name] = f'{NEPAL}/{fname}.tif'
    if os.path.exists(FILES[name]): print('exists:', fname); continue
    f, p = acq(aoi, e['flood'], e['orbit']), acq(aoi, e['pre'], e['orbit'])
    d = ee.Date(e['s2']); s2im = S2.filterBounds(aoi).filterDate(d, d.advance(1, 'day')).mosaic()
    dry = (S2.filterBounds(aoi).filterDate('2025-02-01', '2025-04-30').filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
           .map(lambda im: im.normalizedDifference(['B3', 'B11']).updateMask(clear(im))))
    stack = ee.Image.cat([f.select(['VV', 'VH'], ['flood_VV', 'flood_VH']), p.select(['VV', 'VH'], ['pre_VV', 'pre_VH']),
                          slope.rename('slope'), hand.rename('hand'), layover_shadow(f, aoi),
                          lab(s2im.normalizedDifference(['B3', 'B11']), clear(s2im)).rename('s2_water'),
                          lab(dry.median(), dry.count().gt(0)).rename('dry_water')]).toFloat().clip(aoi)
    t = ee.batch.Export.image.toDrive(image=stack, description=f'{name}_2025', folder='Paper2_flood_nepal',
            fileNamePrefix=fname, region=aoi, scale=10, crs='EPSG:32645', maxPixels=1e10)
    t.start(); tasks.append(t)
while any(t.status()['state'] in ('READY', 'RUNNING') for t in tasks):
    print(' | '.join(f"{t.status()['description']}: {t.status()['state']}" for t in tasks)); time.sleep(30)
for t in tasks: print(t.status()['description'], t.status()['state'], t.status().get('error_message', ''))
for _ in range(20):
    if all(os.path.exists(f) for f in FILES.values()): break
    time.sleep(15)
for f in FILES.values(): assert os.path.exists(f), f'not in Drive yet: {f} - wait a minute and rerun this cell'
print('stacks ready')

## 2. Load stacks, normal water (JRC ≥ 80 %) and the river belt (JRC max extent)

In [ ]:
BANDS = ['flood_VV', 'flood_VH', 'pre_VV', 'pre_VH', 'slope', 'hand', 'ls_mask', 's2_water', 'dry_water']
JRC = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
D = {}
for name, f in FILES.items():
    with rasterio.open(f) as s:
        b = dict(zip(BANDS, s.read().astype('float32'))); prof = s.profile
    shape = b['flood_VV'].shape; T = prof['transform']
    valid = np.ones(shape, bool)
    for k in ['flood_VV', 'flood_VH', 'pre_VV', 'pre_VH']: valid &= np.isfinite(b[k]) & (b[k] != 0)
    valid = binary_erosion(valid, iterations=3)
    grid = {'dimensions': {'width': shape[1], 'height': shape[0]}, 'crsCode': prof['crs'].to_string(),
            'affineTransform': {'scaleX': T.a, 'shearX': T.b, 'translateX': T.c, 'shearY': T.d, 'scaleY': T.e, 'translateY': T.f}}
    px = lambda img, band: ee.data.computePixels({'expression': img, 'fileFormat': 'NUMPY_NDARRAY', 'grid': grid})[band]
    occ = px(JRC.select('occurrence').unmask(0).toUint8(), 'occurrence')
    belt = px(JRC.select('max_extent').unmask(0).toUint8(), 'max_extent') == 1
    D[name] = dict(b=b, prof=prof, shape=shape, valid=valid, normal=occ >= 80, perm=occ >= 80, belt=belt)
    lab_ok = valid & (b['s2_water'] != 255)
    print(f"{name}: {valid.sum() / 1e4:.0f} km² valid, S2 labelled {lab_ok.sum() / 1e4:.0f} km², "
          f"S2 water {(lab_ok & (b['s2_water'] == 1)).sum() / 1e4:.1f} km²")

## 3. Input scaling (same as training) and inference with all 12 models

In [ ]:
ROOT = '/content/sen1floods11'; os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(f'{ROOT}/S1Hand'):
    for p in ['data/flood_events/HandLabeled/S1Hand', 'splits/flood_handlabeled']:
        subprocess.run(['gsutil', '-m', '-q', 'cp', '-r', f'gs://sen1floods11/v1.1/{p}', f'{ROOT}/'], check=True)
CACHE = f'{BASE}/02_terrain/terrain_chips'
def prep(sar, slope_, hand_):
    sar = np.clip(np.nan_to_num(sar, nan=-50), -50, 1)
    t = np.stack([np.clip(np.nan_to_num(slope_), 0, 60), np.log1p(np.clip(np.nan_to_num(hand_), 0, 200))])
    return np.concatenate([sar, t]).astype('float32')
ids = [f.replace('_S1Hand.tif', '') for f in pd.read_csv(f'{ROOT}/flood_handlabeled/flood_train_data.csv', header=None)[0]]
xs = []
for c in ids:
    with rasterio.open(f'{ROOT}/S1Hand/{c}_S1Hand.tif') as s: sar = s.read().astype('float32')
    tt = np.load(f'{CACHE}/{c}.npy'); xs.append(prep(sar, tt[0], tt[1]))
xs = np.stack(xs); MEAN, STD = xs.mean((0, 2, 3)), xs.std((0, 2, 3)) + 1e-6; del xs

@torch.no_grad()
def predict(model, x, tile=512, stride=448, batch=8):
    model.eval(); C_, H_, W_ = x.shape
    Hp = max(tile, math.ceil((H_ - tile) / stride) * stride + tile); Wp = max(tile, math.ceil((W_ - tile) / stride) * stride + tile)
    xp = np.pad(x, ((0, 0), (0, Hp - H_), (0, Wp - W_)), mode='edge')
    prob = np.zeros((Hp, Wp), 'float32'); cnt = np.zeros((Hp, Wp), 'float32')
    pos = [(r, c) for r in range(0, Hp - tile + 1, stride) for c in range(0, Wp - tile + 1, stride)]
    for i in range(0, len(pos), batch):
        pb = pos[i:i + batch]
        p = torch.softmax(model(torch.tensor(np.stack([xp[:, r:r+tile, c:c+tile] for r, c in pb])).to(DEV)), 1)[:, 1].cpu().numpy()
        for (r, c), pi in zip(pb, p): prob[r:r+tile, c:c+tile] += pi; cnt[r:r+tile, c:c+tile] += 1
    return (prob / cnt)[:H_, :W_] > 0.5

PRED = {}
for name, d in D.items():
    b = d['b']
    for date in ['flood', 'pre']:
        x = (prep(np.stack([b[f'{date}_VV'], b[f'{date}_VH']]), b['slope'], b['hand']) - MEAN[:, None, None]) / STD[:, None, None]
        x[:, ~d['valid']] = 0
        for (m, ver, s), f in WEIGHTS.items():
            net = smp.Unet('resnet34', encoder_weights=None, in_channels=len(CH[m]), classes=2).to(DEV)
            net.load_state_dict(torch.load(f, map_location=DEV))
            PRED[(name, date, m, ver, s)] = predict(net, x[CH[m]]) & d['valid']
    print('done', name)

## 4. Koshi 2025: accuracy against same-day Sentinel-2 (8 Oct)
- **Water extent:** water on 8 Oct vs S2 water.
- **New water:** water on 8 Oct minus JRC ≥ 80 % vs (S2 water and not dry-season S2 water), the definition chosen on the 2024 event.

In [ ]:
def score(pred, truth, mask):
    p, t = pred[mask], truth[mask]
    tp, fp, fn = (p & t).sum(), (p & ~t).sum(), (~p & t).sum()
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return dict(IoU=tp / max(tp + fp + fn, 1), precision=pr, recall=rc)

k = D['koshi_terai']; b = k['b']
m1 = k['valid'] & (b['s2_water'] != 255); t1 = b['s2_water'] == 1
m2 = m1 & (b['dry_water'] != 255); t2 = (b['s2_water'] == 1) & (b['dry_water'] == 0)
rows = []
thr = b['flood_VV'] < -15
cd = thr & (b['flood_VV'] - b['pre_VV'] < -3) & ~k['perm']
rows.append({'method': 'threshold (VV < -15 dB)', 'seed': 0, **{f'extent {kk}': v for kk, v in score(thr, t1, m1).items()},
             **{f'new {kk}': v for kk, v in score(thr & ~k['normal'], t2, m2).items()}})
rows.append({'method': 'change detection', 'seed': 0, **{f'new {kk}': v for kk, v in score(cd, t2, m2).items()}})
for (m, ver, s) in WEIGHTS:
    w = PRED[('koshi_terai', 'flood', m, ver, s)]
    rows.append({'method': f'{m} {ver}', 'seed': s, **{f'extent {kk}': v for kk, v in score(w, t1, m1).items()},
                 **{f'new {kk}': v for kk, v in score(w & ~k['normal'], t2, m2).items()}})
kres = pd.DataFrame(rows); kres.to_csv(f'{OUT}/koshi_2025_scores_per_seed.csv', index=False)
cols = ['extent IoU', 'extent precision', 'extent recall', 'new IoU', 'new precision', 'new recall']
g = kres.groupby('method', sort=False)[cols]
ksum = (g.mean().round(3).astype(str) + ' ± ' + g.std().fillna(0).round(3).astype(str))
print(f'scored: extent {m1.sum() / 1e4:.0f} km², new water {m2.sum() / 1e4:.0f} km² (S2 new water {(t2 & m2).sum() / 1e4:.1f} km²)')
ksum.to_csv(f'{OUT}/koshi_2025_summary.csv'); ksum

## 5. Kathmandu 2025: water accuracy (vs S2 11 Oct) and false alarms
The flood had receded by 9 Oct, so new water (9 Oct vs 27 Sep, outside permanent water) is treated as false alarm, as for 2024.

In [ ]:
k = D['kathmandu']; b = k['b']
m1 = k['valid'] & (b['s2_water'] != 255); t1 = b['s2_water'] == 1
ls = b['ls_mask'] > 0
rows = []
thr = b['flood_VV'] < -15
cd = thr & (b['flood_VV'] - b['pre_VV'] < -3) & ~k['perm'] & k['valid']
rows.append({'method': 'change detection', 'seed': 0, 'false new water km²': cd.sum() / 1e4,
             'in layover/shadow km²': (cd & ls).sum() / 1e4})
for (m, ver, s) in WEIGHTS:
    wf, wp = PRED[('kathmandu', 'flood', m, ver, s)], PRED[('kathmandu', 'pre', m, ver, s)]
    fa = wf & ~wp & ~k['perm']
    rows.append({'method': f'{m} {ver}', 'seed': s, **{f'extent {kk}': v for kk, v in score(wf, t1, m1).items()},
                 'false new water km²': fa.sum() / 1e4, 'in layover/shadow km²': (fa & ls).sum() / 1e4})
kt = pd.DataFrame(rows); kt.to_csv(f'{OUT}/kathmandu_2025_per_seed.csv', index=False)
cols = ['extent IoU', 'extent precision', 'extent recall', 'false new water km²', 'in layover/shadow km²']
g = kt.groupby('method', sort=False)[cols]
ktsum = (g.mean().round(3).astype(str) + ' ± ' + g.std().fillna(0).round(3).astype(str))
print(f'S2 labelled {m1.sum() / 1e4:.0f} km², S2 water {(m1 & t1).sum() / 1e4:.1f} km²')
ktsum.to_csv(f'{OUT}/kathmandu_2025_summary.csv'); ktsum

## 6. Koshi 2025 exposure (final model: B fine-tuned 2024, 3-seed majority)
Same method as 2024: patches < 0.2 ha removed, flooding outside the river belt, buildings and people (Nepal only) within 100 m. Uses the exposure layers exported in notebook 09.

In [ ]:
k = D['koshi_terai']; shape = k['shape']; T = k['prof']['transform']
EXP = f'{NEPAL}/koshi_terai_exposure_layers.tif'
assert os.path.exists(EXP), 'run notebook 09 first (exposure layers)'
lay = np.zeros((2,) + shape, 'float32')
with rasterio.open(EXP) as src:
    for i in range(2):
        reproject(rasterio.band(src, i + 1), lay[i], dst_transform=T, dst_crs=k['prof']['crs'], resampling=Resampling.nearest)
bld, pop = lay
H2, W2 = (shape[0] // 10) * 10, (shape[1] // 10) * 10
blk = lambda a: a[:H2, :W2].reshape(H2 // 10, 10, W2 // 10, 10).sum((1, 3))
P, Bc = blk(np.where(k['valid'], pop, 0)), blk(np.where(k['valid'], bld, 0))
ppb = np.zeros(shape, 'float32')
ppb[:H2, :W2] = np.repeat(np.repeat(np.divide(P, Bc, out=np.zeros_like(P), where=Bc > 0), 10, 0), 10, 1)

def despeckle(m, min_px=20):
    l_, n = cc_label(m, structure=np.ones((3, 3))); sz = np.bincount(l_.ravel()); keep = sz >= min_px; keep[0] = False
    return keep[l_]
votes = sum(PRED[('koshi_terai', 'flood', 'B_sar_terrain', 'fine-tuned 2024', s)].astype('uint8') for s in [0, 1, 2])
maps = {'B fine-tuned (final)': (votes >= 2) & ~k['normal'] & k['valid'],
        'threshold (VV < -15 dB)': (k['b']['flood_VV'] < -15) & ~k['normal'] & k['valid']}
rows = []
for name_, fm in maps.items():
    for part, mm in [('all new water', despeckle(fm)), ('outside river belt', despeckle(fm) & ~k['belt'])]:
        nm = binary_dilation(mm, iterations=10) & k['valid']
        rows.append({'flood map': name_, 'part': part, 'km²': round(mm.sum() / 1e4, 1),
                     'buildings within 100 m': int(bld[nm].sum()), 'people within 100 m': int((bld * ppb)[nm].sum())})
exp25 = pd.DataFrame(rows); exp25.to_csv(f'{OUT}/koshi_2025_exposure.csv', index=False); exp25

## 7. Figure: Koshi 2025

In [ ]:
k = D['koshi_terai']; b = k['b']; st = 3
fm = maps['B fine-tuned (final)']
m2 = k['valid'] & (b['s2_water'] != 255) & (b['dry_water'] != 255); t2 = (b['s2_water'] == 1) & (b['dry_water'] == 0)
fig, ax = plt.subplots(1, 3, figsize=(13, 4.8))
ax[0].imshow(np.where(k['valid'], np.clip(b['flood_VV'], -25, 0), np.nan)[::st, ::st], cmap='gray')
ax[0].set_title('(a) Sentinel-1 VV, 8 Oct 2025', loc='left', fontsize=10)
im = np.ones(shape + (3,)); im[~k['valid']] = 0.92; im[k['normal'] & k['valid']] = [0.70, 0.78, 0.88]; im[fm] = [0.16, 0.47, 0.84]
ax[1].imshow(im[::st, ::st]); ax[1].set_title('(b) New water, final model (blue);\n     normal water (light blue)', loc='left', fontsize=10)
im2 = np.full(shape + (3,), 0.85); im2[m2] = 1
im2[m2 & fm & t2] = [0.11, 0.69, 0.48]; im2[m2 & fm & ~t2] = [0.89, 0.29, 0.28]; im2[m2 & ~fm & t2] = [0.16, 0.47, 0.84]
ax[2].imshow(im2[::st, ::st]); ax[2].set_title('(c) vs Sentinel-2, same day: correct (green),\n     false (red), missed (blue), no label (grey)', loc='left', fontsize=10)
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(f'{BASE}/09_exposure_figures/figures/fig5_koshi_2025.png', dpi=300, bbox_inches='tight'); plt.show()